# Adv-VLM Cookbook

In [ ]:
import torch
from src.utils import resolve_device

device = resolve_device('auto')

print("Using device:", device)

The Huggingface pipeline for VLM generation.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
model = AutoModelForMultimodalLM.from_pretrained("llava-hf/llava-1.5-7b-hf").to(device)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

## Using the Model Interface

Load the LLaVA model wrapper.

In [ ]:
from src.model import LLaVA

llava = LLaVA(device='mps')

Load one example image.

In [ ]:
from transformers.image_utils import load_image
from src.image import raw2resized

image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG")

resized = raw2resized(image)[0]
resized

Use LLaVA to generate.

In [ ]:
llava.gen(resized, "what is in this picture?")

### STL-10 classification evaluation

Options provided in the question. Accuracy is near 1.

In [ ]:
from src.bench.imagenette import Imagenette

imagenette = Imagenette()

imagenette.eval_classify_lp(
    llava,
    question="What is the main object in this photo?\nOptions: tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute.",
    answer_priming="The main object in this photo:",
    batch_size=4,
    limit=64,
    shuffle=True
)

Options not provided in the question. Accuracy drops.

In [ ]:
from src.bench.imagenette import Imagenette

imagenette = Imagenette()

imagenette.eval_classify_lp(
    llava,
    question="What is the main object in this photo?",
    answer_priming="The main object in this photo:",
    batch_size=4,
    limit=64,
    shuffle=True
)